In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from torch_geometric.nn import SAGEConv

In [2]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [4]:
data = np.load(
    "nft_graph_dataset.npz",
    allow_pickle=True
)

print(data.files)

['src', 'dst', 'timestamps', 'edge_feat', 'labels', 'train_mask', 'val_mask', 'test_mask']


In [5]:
src = torch.tensor(
    data["src"],
    dtype=torch.long
)

dst = torch.tensor(
    data["dst"],
    dtype=torch.long
)

edge_index = torch.stack(
    [src, dst],
    dim=0
)

edge_feat = torch.tensor(
    data["edge_feat"],
    dtype=torch.float32
)

labels = torch.tensor(
    data["labels"],
    dtype=torch.float32
)

train_mask = torch.tensor(data["train_mask"])
val_mask = torch.tensor(data["val_mask"])
test_mask = torch.tensor(data["test_mask"])

num_nodes = int(
    max(src.max(), dst.max())
) + 1

print("Nodes:", num_nodes)
print("Edges:", edge_index.shape[1])

Nodes: 333077
Edges: 2713386


In [6]:
print()

print("Train Fraud:",
      labels[train_mask].sum().item())

print("Val Fraud:",
      labels[val_mask].sum().item())

print("Test Fraud:",
      labels[test_mask].sum().item())


Train Fraud: 15818.0
Val Fraud: 1684.0
Test Fraud: 1451.0


In [7]:
class GraphSAGEEdgeClassifier(nn.Module):

    def __init__(
        self,
        num_nodes,
        edge_feat_dim,
        embedding_dim=64,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.node_embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.sage1 = SAGEConv(
            embedding_dim,
            hidden_dim
        )

        self.sage2 = SAGEConv(
            hidden_dim,
            hidden_dim
        )

        self.dropout = nn.Dropout(dropout)

        self.edge_mlp = nn.Sequential(
            nn.Linear(
                hidden_dim * 2 + edge_feat_dim,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                hidden_dim // 2
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim // 2,
                1
            )
        )

    def forward(
        self,
        edge_index,
        src,
        dst,
        edge_feat
    ):

        x = self.node_embedding.weight

        x = self.sage1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.dropout(x)

        x = self.sage2(
            x,
            edge_index
        )

        src_emb = x[src]
        dst_emb = x[dst]

        z = torch.cat(
            [
                src_emb,
                dst_emb,
                edge_feat
            ],
            dim=1
        )

        logits = self.edge_mlp(z)

        return logits.squeeze(-1)

In [8]:
model = GraphSAGEEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat.shape[1]
).to(DEVICE)

print(model)

GraphSAGEEdgeClassifier(
  (node_embedding): Embedding(333077, 64)
  (sage1): SAGEConv(64, 128, aggr=mean)
  (sage2): SAGEConv(128, 128, aggr=mean)
  (dropout): Dropout(p=0.3, inplace=False)
  (edge_mlp): Sequential(
    (0): Linear(in_features=265, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)
